# Chest X-ray — SMOKE TEST (bug-check before the 100-epoch run)

**Purpose:** fast run that exercises EVERY stage incl. the 3 fixed bugs (ViT load, XAI memory, normalized SRC + real masks). 3 models × 3 epochs. The NUMBERS are garbage (3 epochs) — you are only checking that **nothing crashes** and every artifact is produced.

If all green -> open `colab_full.ipynb` for the real 100-epoch run.

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone / update code

In [ ]:
import os
REPO='https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    !git clone $REPO /content/Chest_Xray
else:
    !git -C /content/Chest_Xray pull
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — CRASH-SAFETY: point results/ AT Google Drive (BEFORE anything writes)
Every checkpoint, JSON and figure now lands on Drive the INSTANT it is written. A disconnect no longer loses results — reconnect, re-run, and `--resume` skips finished models.

In [ ]:
import os, shutil
DRIVE='/content/drive/MyDrive/cxr_data'
RES=f'{DRIVE}/results'; BACKUP=f'{DRIVE}/backup'
os.makedirs(RES,exist_ok=True); os.makedirs(BACKUP,exist_ok=True)
if os.path.islink('results'): os.remove('results')
elif os.path.isdir('results'): shutil.rmtree('results')
os.symlink(RES,'results')
print('results ->', os.path.realpath('results')); print('backup  ->', BACKUP)

## Cell 4 — unzip data to fast local disk (slow, once per session)

In [ ]:
import os, glob
DRIVE='/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw',exist_ok=True)
for z in glob.glob(f'{DRIVE}/*.zip'):
    print('unzipping', os.path.basename(z), flush=True)
    !unzip -q -o "$z" -d data/raw
print('folders:', os.listdir('data/raw'))

## Cell 5 — bring the manifest

In [ ]:
import os, shutil, pandas as pd
DRIVE='/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE}/manifest.csv'):
    shutil.copy(f'{DRIVE}/manifest.csv','data/manifest.csv'); print('manifest copied from Drive')
else:
    !python scripts/build_manifest.py
# VERIFY the manifest has the mask_path column (real lung masks). If False, you uploaded the OLD manifest.
_df=pd.read_csv('data/manifest.csv')
print('rows:', len(_df), '| mask_path column present:', 'mask_path' in _df.columns)
if 'mask_path' not in _df.columns:
    print('>>> STOP: re-upload the NEW manifest.csv to Drive (it must have mask_path for real masks).')
else:
    print('   rows with real mask:', (_df['mask_path'].fillna('')!='').sum())

## Cell 6 — install deps + GPU check (GPU must NOT be empty)

In [ ]:
!pip -q install -r requirements.txt
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 7 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df=pd.read_csv('data/manifest.csv')
missing=[p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
print('>>> OK: continue.' if not missing else f'>>> STOP. example: {missing[0]} ; folders: {os.listdir("data/raw")}')

## SMOKE RUN — 3 models, 3 epochs, ALL stages
Chosen to trigger every fixed bug: **vit** (load crash), **densenet201** (XAI out-of-memory on big backbone), **lenet5** (invalid-cert path + fast). Backup goes to a SEPARATE smoke folder so it never touches your real run.

In [ ]:
import os; os.chdir('/content/Chest_Xray')
MODELS='densenet201 vit lenet5'
BACKUP='/content/drive/MyDrive/cxr_data/backup_smoke'
!python -u scripts/run_pipeline.py --models $MODELS --epochs 3 --batch-size 32 --per-class 40 --backup-dir $BACKUP

## VERIFY — all stages ran + the 3 fixes work

In [ ]:
import os, json
ok=True
for f in ['cross_source.json','c3_coupling.json','src_vs_baselines.json','lomo.json']:
    e=os.path.exists(f'results/{f}'); ok=ok and e; print(f, 'OK' if e else 'MISSING ❌')
# fix 1: ViT loaded (it has a certificate)
print('vit certificate:', 'OK' if os.path.exists('results/vit/certificate.json') else 'MISSING ❌ (ViT load broke)')
# fix 2: normalized SRC present
c=json.load(open('results/densenet201/certificate.json'))
print('normalized SRC field:', 'OK' if 'normalization' in c else 'MISSING ❌', '| src=', round(c.get('src',float('nan')),3), 'raw_mean=', round(c.get('src_raw_mean',float('nan')),3))
# fix 3: real masks -> in_lung is a NUMBER not nan
x=json.load(open('results/densenet201/xai.json'))
il=x['grad_cam']['in_lung']
print('in_lung (real masks):', 'OK ('+str(round(il,3))+')' if il==il else 'nan ❌ (masks not flowing — check manifest mask_path)')
print('figures:', os.listdir('results/figures') if os.path.isdir('results/figures') else '❌')
print('tables :', os.listdir('results/tables') if os.path.isdir('results/tables') else '❌')
print('\n>>> ALL GREEN — proceed to colab_full.ipynb' if ok else '\n>>> something failed — paste output')

## CLEAN UP smoke artifacts (so they don't interfere with the real run)

In [ ]:
import shutil
shutil.rmtree('/content/drive/MyDrive/cxr_data/backup_smoke', ignore_errors=True)
# NOTE: leave results/ — the clean-slate cell in colab_full.ipynb wipes it before the real run.
print('removed backup_smoke. Now open colab_full.ipynb (its clean-slate cell handles results/).')